In [ ]:
%load_ext autoreload
%autoreload 2

import featuregraph as fg

In [ ]:
smooth_window = 125

df = fg.datasets.bidmc(subject=1)

# Construction parameters
subject_id = 1
sampling_rate_hz = 125
smooth_window = 100

# Preserve the input DataFrame
df = df.copy()

# Observation identity and time
df['subject_id'] = subject_id
df['sample_index'] = df.index
df['time_seconds'] = df['sample_index'] / sampling_rate_hz

# Construct the max-then-mean respiratory envelope
df['respiration_smooth'] = (
    df['respiration']
    .rolling(smooth_window)
    .max()
    .rolling(smooth_window)
    .mean()
    .shift(-smooth_window)
)

# Identify samples where the constructed envelope is defined
df['respiration_smooth_valid'] = df['respiration_smooth'].notna()

# Change in the constructed envelope
df['respiration_change'] = df['respiration_smooth'].diff()

# Mutually exclusive respiratory states
df['respiration_rising'] = (
    df['respiration_smooth_valid']
    & df['respiration_change'].gt(0)
)

df['respiration_falling'] = (
    df['respiration_smooth_valid']
    & df['respiration_change'].lt(0)
)

df['respiration_inactive'] = (
    df['respiration_smooth_valid']
    & df['respiration_change'].eq(0)
)

# Events marking entry into and exit from the rising state
df['enter_respiration_rising'] = (
    df['respiration_rising']
    .astype(int)
    .diff()
    .eq(1)
)

df['exit_respiration_rising'] = (
    df['respiration_rising']
    .astype(int)
    .diff()
    .eq(-1)
)

# Candidate peak-to-peak identity
df['exit_respiration_rising_id'] = (
    df['exit_respiration_rising']
    .cumsum()
)

# Validate that every valid constructed sample has exactly one state
state_columns = [
    'respiration_rising',
    'respiration_falling',
    'respiration_inactive',
]

state_count = df[state_columns].sum(axis=1)

assert state_count[df['respiration_smooth_valid']].eq(1).all()

In [ ]:
df['respiration_smooth_valid'].value_counts()

In [ ]:
import numpy as np
import pandas as pd


# -------------------------------------------------------------------------
# Prepare sample-level fields used during object construction
# -------------------------------------------------------------------------

group_column = 'exit_respiration_rising_id'

# Within each candidate, this becomes True after the trough/turn event.
df['entered_rising_in_candidate'] = (
    df.groupby(group_column)['enter_respiration_rising']
    .cumsum()
    .gt(0)
)

# Separate inactive samples by their location within the cycle.
df['trough_inactive'] = (
    df['respiration_inactive']
    & ~df['entered_rising_in_candidate']
)

df['peak_inactive'] = (
    df['respiration_inactive']
    & df['entered_rising_in_candidate']
)

# Event landmarks. Non-event rows remain NaN.
df['start_peak_sample'] = df['sample_index'].where(
    df['exit_respiration_rising']
)

df['start_peak_time'] = df['time_seconds'].where(
    df['exit_respiration_rising']
)

df['trough_sample'] = df['sample_index'].where(
    df['enter_respiration_rising']
)

df['trough_time'] = df['time_seconds'].where(
    df['enter_respiration_rising']
)


# -------------------------------------------------------------------------
# Construct one row per peak-to-peak candidate
# -------------------------------------------------------------------------

summarydf = (
    df.groupby(group_column, sort=True)
    .agg(
        subject_id=('subject_id', 'first'),

        # Candidate identity and observed extent
        first_observed_sample=('sample_index', 'first'),
        last_observed_sample=('sample_index', 'last'),
        first_observed_time=('time_seconds', 'first'),
        last_observed_time=('time_seconds', 'last'),

        # Structural landmarks
        start_peak_sample=('start_peak_sample', 'first'),
        start_peak_time=('start_peak_time', 'first'),
        trough_sample=('trough_sample', 'first'),
        trough_time=('trough_time', 'first'),

        # State durations in samples
        respiration_rising_samples=('respiration_rising', 'sum'),
        respiration_falling_samples=('respiration_falling', 'sum'),
        respiration_inactive_samples=('respiration_inactive', 'sum'),
        trough_inactive_samples=('trough_inactive', 'sum'),
        peak_inactive_samples=('peak_inactive', 'sum'),

        # Expected event counts
        enter_rising_count=('enter_respiration_rising', 'sum'),
        exit_rising_count=('exit_respiration_rising', 'sum'),

        # Raw-signal measurements
        raw_minimum=('respiration', 'min'),
        raw_maximum=('respiration', 'max'),

        # Envelope measurements
        smooth_minimum=('respiration_smooth', 'min'),
        smooth_maximum=('respiration_smooth', 'max'),

        # Data-quality information
        observed_samples=('sample_index', 'size'),
        valid_smooth_samples=('respiration_smooth_valid', 'sum'),
    )
    .reset_index()
    .rename(columns={group_column: 'candidate_id'})
)


# -------------------------------------------------------------------------
# Add the right-hand peak boundary
#
# Each group begins at an exit-rising event. Therefore, the start of the
# next group is the right-hand peak boundary of the current candidate.
# -------------------------------------------------------------------------

summarydf['end_peak_sample'] = (
    summarydf.groupby('subject_id')['start_peak_sample']
    .shift(-1)
)

summarydf['end_peak_time'] = (
    summarydf.groupby('subject_id')['start_peak_time']
    .shift(-1)
)


# -------------------------------------------------------------------------
# Convert sample counts to physical time
# -------------------------------------------------------------------------

duration_columns = {
    'respiration_rising_samples': 'respiration_rising_seconds',
    'respiration_falling_samples': 'respiration_falling_seconds',
    'respiration_inactive_samples': 'respiration_inactive_seconds',
    'trough_inactive_samples': 'trough_inactive_seconds',
    'peak_inactive_samples': 'peak_inactive_seconds',
}

for sample_column, seconds_column in duration_columns.items():
    summarydf[seconds_column] = (
        summarydf[sample_column] / sampling_rate_hz
    )


# -------------------------------------------------------------------------
# Derived intrinsic properties
# -------------------------------------------------------------------------

summarydf['period_samples'] = (
    summarydf['end_peak_sample']
    - summarydf['start_peak_sample']
)

summarydf['period_seconds'] = (
    summarydf['end_peak_time']
    - summarydf['start_peak_time']
)

summarydf['raw_amplitude'] = (
    summarydf['raw_maximum']
    - summarydf['raw_minimum']
)

summarydf['smooth_amplitude'] = (
    summarydf['smooth_maximum']
    - summarydf['smooth_minimum']
)

summarydf['active_samples'] = (
    summarydf['respiration_rising_samples']
    + summarydf['respiration_falling_samples']
)

# Fraction of directional motion spent rising.
# A value of 0.5 represents equal rising and falling duration.
summarydf['rising_fraction'] = np.where(
    summarydf['active_samples'].gt(0),
    summarydf['respiration_rising_samples']
    / summarydf['active_samples'],
    np.nan,
)

# Signed temporal symmetry:
#     0  = equal rising and falling durations
#     >0 = longer rising phase
#     <0 = longer falling phase
summarydf['temporal_symmetry'] = np.where(
    summarydf['active_samples'].gt(0),
    (
        summarydf['respiration_rising_samples']
        - summarydf['respiration_falling_samples']
    )
    / summarydf['active_samples'],
    np.nan,
)


# -------------------------------------------------------------------------
# Completeness and structural interpretation
# -------------------------------------------------------------------------

summarydf['has_start_peak'] = summarydf['start_peak_time'].notna()
summarydf['has_end_peak'] = summarydf['end_peak_time'].notna()
summarydf['has_single_trough_turn'] = (
    summarydf['enter_rising_count'].eq(1)
)

summarydf['is_complete'] = (
    summarydf['has_start_peak']
    & summarydf['has_end_peak']
    & summarydf['has_single_trough_turn']
)

summarydf['all_samples_have_valid_envelope'] = (
    summarydf['valid_smooth_samples']
    .eq(summarydf['observed_samples'])
)

summarydf['is_structurally_valid'] = (
    summarydf['is_complete']
    & summarydf['respiration_rising_samples'].gt(0)
    & summarydf['respiration_falling_samples'].gt(0)
    & summarydf['period_samples'].gt(0)
)


# -------------------------------------------------------------------------
# Preserve reasons for incomplete or unusual objects
# -------------------------------------------------------------------------

def classify_candidate(row):
    reasons = []

    if not row['has_start_peak']:
        reasons.append('missing_start_peak')

    if not row['has_end_peak']:
        reasons.append('missing_end_peak')

    if row['enter_rising_count'] == 0:
        reasons.append('missing_trough_turn')
    elif row['enter_rising_count'] > 1:
        reasons.append('multiple_trough_turns')

    if row['respiration_rising_samples'] == 0:
        reasons.append('no_rising_samples')

    if row['respiration_falling_samples'] == 0:
        reasons.append('no_falling_samples')

    if not row['all_samples_have_valid_envelope']:
        reasons.append('incomplete_envelope_support')

    return 'valid' if not reasons else ';'.join(reasons)


summarydf['candidate_status'] = summarydf.apply(
    classify_candidate,
    axis=1,
)


# -------------------------------------------------------------------------
# Stable object identity and construction provenance
# -------------------------------------------------------------------------

summarydf['object_id'] = (
    summarydf['subject_id'].astype(str)
    + ':respiration:'
    + summarydf['candidate_id'].astype(str)
)

summarydf['sampling_rate_hz'] = sampling_rate_hz
summarydf['smooth_window_samples'] = smooth_window
summarydf['smooth_window_seconds'] = (
    smooth_window / sampling_rate_hz
)


# Optional publication-facing column order
object_columns = [
    'object_id',
    'subject_id',
    'candidate_id',
    'start_peak_sample',
    'trough_sample',
    'end_peak_sample',
    'start_peak_time',
    'trough_time',
    'end_peak_time',
    'period_samples',
    'period_seconds',
    'respiration_rising_samples',
    'respiration_falling_samples',
    'respiration_inactive_samples',
    'trough_inactive_samples',
    'peak_inactive_samples',
    'respiration_rising_seconds',
    'respiration_falling_seconds',
    'respiration_inactive_seconds',
    'trough_inactive_seconds',
    'peak_inactive_seconds',
    'raw_minimum',
    'raw_maximum',
    'raw_amplitude',
    'smooth_minimum',
    'smooth_maximum',
    'smooth_amplitude',
    'rising_fraction',
    'temporal_symmetry',
    'enter_rising_count',
    'exit_rising_count',
    'is_complete',
    'is_structurally_valid',
    'candidate_status',
    'sampling_rate_hz',
    'smooth_window_samples',
    'smooth_window_seconds',
]

summarydf = summarydf[object_columns]

In [ ]:
complete_objects = summarydf.loc[
    summarydf['is_structurally_valid']
].copy()

In [ ]:
complete_objects